# 05 — Merge Module Data with NSS

Glue the two halves of the project together.

**Inputs (from `data/processed2/`):**
- `dept_year_summary.csv`     — one row per (dept, academic year)
- `nss_exeter_wide.csv`       — one row per (NSS year, subject)
- `dept_to_nss_subject.csv`   — module dept code → NSS subject

**Outputs (to `data/processed2/`):**
- `merged_subject.csv` — one row per (NSS subject × year). Used for the headline charts (3 and 4).
- `merged_dept.csv`    — one row per (dept × year), each row carrying the NSS scores of its parent CAH2 subject.
  *This is new in v2: it gives the model ~2× more rows since multiple Exeter departments map to one CAH2 subject.*

## Year-matching logic

Module-bank uses academic years like `'2022/3'`; NSS uses calendar years like `2023`.
We line them up: NSS 2023 surveys students who studied in academic year 2022/3.

| `academic_year` | `nss_year` |
|---|---|
| 2022/3 | 2023 |
| 2023/4 | 2024 |
| 2024/5 | 2025 |

Earlier academic years (2019/20 → 2021/22) have no NSS counterpart. They stay in the module summary so notebook 06 can use them for the trend chart, but they drop out of the merged satisfaction file.

## 1. Setup

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / 'data' / 'raw').exists() else HERE.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed2'

modules = pd.read_csv(PROCESSED / 'dept_year_summary.csv')
nss     = pd.read_csv(PROCESSED / 'nss_exeter_wide.csv')
mapping = pd.read_csv(PROCESSED / 'dept_to_nss_subject.csv')

print(f'modules : {len(modules):,} rows')
print(f'nss     : {len(nss):,} rows')
print(f'mapping : {len(mapping):,} rows')

modules : 152 rows
nss     : 72 rows
mapping : 36 rows


## 2. Convert academic year → NSS calendar year

In [15]:
def academic_to_nss_year(ay):
    if not isinstance(ay, str) or '/' not in ay:
        return None
    try:
        return int(ay.split('/')[0]) + 1
    except ValueError:
        return None

modules['nss_year'] = modules['academic_year'].apply(academic_to_nss_year)
print(modules[['academic_year', 'nss_year']].drop_duplicates().sort_values('academic_year').to_string(index=False))

academic_year  nss_year
       2019/0      2020
       2020/1      2021
       2021/2      2022
       2022/3      2023
       2023/4      2024
       2024/5      2025


## 3. Sanity-check the mapping

Three things can go wrong:
1. A `dept_code` in the mapping that doesn't exist in the module data.
2. An `nss_subject` in the mapping that doesn't exist in the NSS data.
3. A real `dept_code` that has **no** mapping entry — these silently fall out of the merged file.

In [17]:
real_codes      = set(modules['dept_code'].dropna())
real_subjects   = set(nss['subject'].dropna())
mapped_codes    = set(mapping['dept_code'])
mapped_subjects = set(mapping['nss_subject'])

print(f'Mapping codes not in module data:    {sorted(mapped_codes - real_codes) or "(none)"}')
print(f'Mapping subjects not in NSS data:    {sorted(mapped_subjects - real_subjects) or "(none)"}')
print(f'Module codes with no mapping (will not be merged):')
for c in sorted(real_codes - mapped_codes):
    print(f'  - {c}')

Mapping codes not in module data:    ['BEA', 'BEF', 'MDC', 'MED', 'MLF', 'MLG', 'MLI', 'MLS', 'NUR', 'SHS', 'SPA']
Mapping subjects not in NSS data:    (none)
Module codes with no mapping (will not be merged):
  - BUS
  - CMM
  - EDU
  - EMP
  - LIB


## 4. Build `merged_dept.csv` — one row per (dept × year)  *(NEW in v2)*

Attach the NSS subject onto each dept-year row, then attach the NSS theme scores from the parent subject.
Multiple Exeter depts that map to the same NSS subject (e.g. SOC, SPA, ANT all → `'Sociology, social policy and anthropology'`) all get the same NSS scores — this preserves the dept-level variation in module features that v1 averaged away.

In [19]:
modules_with_subj = modules.merge(mapping, on='dept_code', how='inner')
merged_dept = modules_with_subj.merge(
    nss,
    left_on=['nss_subject', 'nss_year'],
    right_on=['subject', 'nss_year'],
    how='left',
).drop(columns=['subject'])

front = ['dept_code', 'faculty', 'nss_subject', 'academic_year', 'nss_year', 'n_modules']
other = [c for c in merged_dept.columns if c not in front]
merged_dept = merged_dept[front + other]

merged_dept.to_csv(PROCESSED / 'merged_dept.csv', index=False)
n_dept_with_nss = merged_dept['nss_assessment_feedback'].notna().sum()
print(f'Wrote merged_dept.csv: {len(merged_dept):,} rows  ({n_dept_with_nss:,} with NSS match)')
merged_dept.head()

Wrote merged_dept.csv: 136 rows  (72 with NSS match)


,dept_code,faculty,nss_subject,academic_year,nss_year,n_modules,mean_coursework_pct,mean_written_exam_pct,mean_practical_pct,mean_class_size,...,mean_n_summative_items,subject_code,nss_academic_support,nss_assessment_feedback,nss_free_expression,nss_learning_opps,nss_learning_resources,nss_organisation,nss_student_voice,nss_teaching
0,ANT,HASS,"Sociology, social policy and anthropology",2019/0,2020,19.0,85.500000,14.500000,0.000000,30.800000,...,1.700000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ANT,HASS,"Sociology, social policy and anthropology",2020/1,2021,24.0,84.400000,11.600000,4.000000,28.320000,...,1.760000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANT,HASS,"Sociology, social policy and anthropology",2021/2,2022,26.0,90.000000,6.296296,3.703704,28.444444,...,1.851852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANT,HASS,"Sociology, social policy and anthropology",2022/3,2023,34.0,92.857143,1.428571,5.714286,29.485714,...,1.828571,CAH15-01,75.066667,72.014490,60.0,77.725,88.631214,80.875651,62.359266,84.975000
4,ANT,HASS,"Sociology, social policy and anthropology",2024/5,2025,38.0,95.714286,1.904762,2.380952,34.095238,...,1.904762,CAH15-01,79.445763,76.076336,72.7,84.825,93.014442,87.965208,66.747991,90.775594


## 5. Build `merged_subject.csv` — one row per (NSS subject × year)

Aggregate the dept rows up to NSS subject level for the headline charts. We weight by `n_modules` so a small dept doesn't pull a big dept around.

In [21]:
def w_mean(v, w):
    v = pd.Series(v).astype(float); w = pd.Series(w).astype(float)
    ok = v.notna() & w.notna() & (w > 0)
    return (v[ok] * w[ok]).sum() / w[ok].sum() if ok.any() else np.nan

feature_cols = [
    'mean_coursework_pct', 'mean_written_exam_pct', 'mean_practical_pct',
    'mean_class_size',     'mean_scheduled_hours',  'mean_contact_ratio',
    'mean_assess_diversity', 'mean_n_summative_items',
]

def collapse(group):
    w = group['n_modules']
    out = {f: w_mean(group[f], w) for f in feature_cols}
    out['n_modules_total']     = group['n_modules'].sum()
    out['dept_codes_included'] = ','.join(sorted(group['dept_code'].unique()))
    out['faculty']             = group['faculty'].mode().iloc[0] if len(group['faculty'].mode()) else ''
    return pd.Series(out)

subj = (
    modules_with_subj
      .groupby(['nss_subject', 'academic_year', 'nss_year'], dropna=False)
      .apply(collapse, include_groups=False)
      .reset_index()
)

merged_subject = subj.merge(
    nss,
    left_on=['nss_subject', 'nss_year'],
    right_on=['subject', 'nss_year'],
    how='left',
).drop(columns=['subject'])

front = ['nss_subject', 'academic_year', 'nss_year', 'faculty', 'dept_codes_included', 'n_modules_total']
other = [c for c in merged_subject.columns if c not in front]
merged_subject = merged_subject[front + other]

merged_subject.to_csv(PROCESSED / 'merged_subject.csv', index=False)
n_with_nss = merged_subject['nss_assessment_feedback'].notna().sum()
print(f'Wrote merged_subject.csv: {len(merged_subject):,} rows  ({n_with_nss:,} with NSS match)')

Wrote merged_subject.csv: 105 rows  (56 with NSS match)


## 6. Coverage summary

In [23]:
print('Rows in merged_dept    by NSS year:')
print(merged_dept.groupby('nss_year').size())
print('\nRows in merged_subject by NSS year:')
print(merged_subject.groupby('nss_year').size())

Rows in merged_dept    by NSS year:
nss_year
2020    18
2021    22
2022    24
2023    25
2024    22
2025    25
dtype: int64

Rows in merged_subject by NSS year:
nss_year
2020    14
2021    17
2022    18
2023    19
2024    18
2025    19
dtype: int64


Next: **`06_analysis_and_plots.ipynb`** — charts + models on the merged files.